# 01 · AKUDENTAL Panoramic WSP Benchmark (Step-by-Step Validation & Visuals)

Self-contained, resume-safe benchmark notebook using the official public AKUDENTAL folder (`akudental_instances.json` + `images/`, stratified **200 train / 67 val / 66 sealed test**).

**Run cleanly from top to bottom on a Colab GPU (T4 / L4):**
- **Step 1:** Mount Google Drive, locate `dental-wsp` & `AKUDENTAL`, verify GPU (`Device: cuda`), and set permanent Drive checkpoint folder.
- **Step 2 (`akudental_cls` · Stage 1):** Frozen WSP (`K = 0`, all 6 locked replies) on the 67 validation panoramics.
- **Step 3 (`akudental_cls` · Stages 2 & 3):** 3-Arm learning curve (`Frozen WSP` vs. `Random-Init Adam` vs. `WSP Warm-Start Adam`, `K = 2..200`, 5 seeds).
- **Step 4 (`akudental_seg` · Stage 1):** Frozen `pano-wsp82-v1` full mechanistic audit (A/B/C ablations, paired swaps, outside FP share, boundary F1, edge recall, excluded tooth pixels).
- **Step 5 (`akudental_seg` · Stages 2 & 3):** Converged 4-Arm 82-param learning curve (`Frozen WSP`, `Trained Position-Only`, `Random-Init Adam`, `WSP Warm-Start Adam`, `K = 1..200`, 3 seeds, mini-batch `batch_size=32`, `epochs=150`, cosine LR, Drive-checkpointed after every step).
- **Step 6 (Visuals · Frozen Replies):** Deterministic X-ray overlay grid comparing all 6 frozen replies (`A · Intact` vs. `B · Image-Only`).
- **Step 7 (Visuals · 4 Arms):** Side-by-side X-ray overlay comparing all 4 arms at `K = 200` on the exact same validation panoramics.

**Governance:** Held-out test split (`66` films) stays sealed (`split='val'`). Clear outputs before committing to Git.

## Step 1 · Setup, Drive Mount & GPU Check

In [ ]:
from pathlib import Path
import sys

# Auto-detect Google Colab vs. local Windows
if Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")  # Writable so checkpoints & figures persist across disconnects
    DATA_DIR = Path("/content/drive/MyDrive/WSP_Dental_Data/AKUDENTAL")
    candidates = [Path("/content/dental-wsp"), Path("/content/drive/MyDrive/dental-wsp")]
    REPO_DIR = next((p for p in candidates if (p / "benchmarks").is_dir()), candidates[1])
    OUT_ROOT = Path("/content/drive/MyDrive/dental-wsp-runs/akudental")
else:
    REPO_DIR = Path(r"C:\Users\Lenovo\Desktop\dental-wsp")
    DATA_DIR = Path(r"C:\Users\Lenovo\Desktop\WSP_Dental_Data\AKUDENTAL")
    OUT_ROOT = REPO_DIR / "runs/akudental"

assert (REPO_DIR / "benchmarks").is_dir(), (
    f"dental-wsp code folder not found at {REPO_DIR}. "
    "Upload C:/Users/Lenovo/Desktop/dental-wsp to My Drive/dental-wsp (or clone to /content/dental-wsp)."
)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

OUT_ROOT.mkdir(parents=True, exist_ok=True)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from benchmarks.akudental.adapter import verify_akudental_source
from wsp_core.guards import exposure_guard
exposure_guard(DATA_DIR)
print(f"Repo:        {REPO_DIR}")
print(f"Data:        {DATA_DIR}")
print(f"Checkpoints: {OUT_ROOT}")
print(f"Device:      {DEVICE}")
print(verify_akudental_source(DATA_DIR))

## Step 2 · `akudental_cls` Stage 1 — Frozen WSP (`K = 0`, 6 Replies)

In [ ]:
import pandas as pd
from benchmarks.akudental.adapter import load_akudental_cls_public
from wsp_core.evaluation.classification import evaluate_classifier
from wsp_core.schemas import strict_json_loads, verify_response_lock

train_cls = load_akudental_cls_public(DATA_DIR, split="train")
val_cls = load_akudental_cls_public(DATA_DIR, split="val")
print(f"akudental_cls cohorts: train={len(train_cls)} (pos={sum(s.label for s in train_cls)}), val={len(val_cls)} (pos={sum(s.label for s in val_cls)})")

cls_design = REPO_DIR / "benchmarks/akudental/locks/akudental_cls.design.lock.json"
cls_resp = REPO_DIR / "benchmarks/akudental/locks/akudental_cls.responses.lock.json"
cls_d_bytes = cls_design.read_bytes()
frozen_cls = verify_response_lock(strict_json_loads(cls_d_bytes), cls_d_bytes, strict_json_loads(cls_resp.read_bytes()), "cls-lenet2d-v1")

cls_stage1_df = pd.DataFrame([{"Reply": slot, **evaluate_classifier(p, val_cls)} for slot, p in frozen_cls.items()]).set_index("Reply")
cls_stage1_df.to_csv(OUT_ROOT / "akudental_cls_stage1_frozen.csv")
display(cls_stage1_df[["roc_auc", "pr_auc", "accuracy", "sensitivity", "specificity", "margin_ablated_roc_auc", "sign_flip_flagged"]].round(4))

## Step 3 · `akudental_cls` Stages 2 & 3 — 3-Arm Learning Curve (`K = 2..200`, 5 Seeds)

In [ ]:
import matplotlib.pyplot as plt
from wsp_core.training.runner import plot_classification_learning_curve, run_classification_three_arm_validation

CLS_OUT = OUT_ROOT / "akudental_cls"
cls_runs, cls_summary = run_classification_three_arm_validation(
    task="akudental_cls",
    train_samples=train_cls,
    val_samples=val_cls,
    design_lock_path=cls_design,
    responses_lock_path=cls_resp,
    budgets=[2, 4, 8, 16, 32, 64, len(train_cls)],
    seeds=[0, 1, 2, 3, 4],
    epochs=200,
    lr=5e-3,
    device=DEVICE,
    out_dir=CLS_OUT,
    resume=True,
)
display(cls_summary.round(4))
fig = plot_classification_learning_curve(cls_summary, task="akudental_cls")
plt.show()
plt.close(fig)
print(f"Saved akudental_cls runs, summary CSV, and PNG to {CLS_OUT}")

## Step 4 · `akudental_seg` Stage 1 — Frozen WSP Mechanistic Audit (`K = 0`, 6 Replies)

In [ ]:
import pandas as pd
from benchmarks.akudental.adapter import load_akudental_seg_public
from wsp_core.evaluation.segmentation import evaluate_segmentation_paired_audit, evaluate_segmentation_swaps
from wsp_core.schemas import strict_json_loads, verify_response_lock

val_seg = load_akudental_seg_public(DATA_DIR, split="val")
seg_design = REPO_DIR / "benchmarks/akudental/locks/akudental_seg.design.lock.json"
seg_resp = REPO_DIR / "benchmarks/akudental/locks/akudental_seg.responses.lock.json"
seg_d_bytes = seg_design.read_bytes()
frozen_seg = verify_response_lock(strict_json_loads(seg_d_bytes), seg_d_bytes, strict_json_loads(seg_resp.read_bytes()), "pano-wsp82-v1")

seg_rows = []
for slot, payload in frozen_seg.items():
    sw = evaluate_segmentation_swaps(payload, val_seg)
    au = evaluate_segmentation_paired_audit(payload, val_seg)
    seg_rows.append({
        "Reply": slot,
        "Intact Dice (%)": sw["full_cohort_dice_a_pct"],
        "Image-only Dice (%)": sw["full_cohort_dice_b_pct"],
        "Position-only Dice (%)": sw["full_cohort_dice_c_pct"],
        "Swap loss A (pp)": sw["intact"]["swap_loss_pp"],
        "Swap groups": f"{sw['groups']}/{sw['total_groups']}",
        "Outside FP share (%)": au["outside_fp_reduction_share_pct"],
        "Delta BF1 (pp)": au["bf1_gain_pp"],
        "Delta edge recall (pp)": au["edge_recall_gain_pp"],
        "Box IoU A (%)": au["box_iou_a_pct"],
        "Excluded tooth pixels A (%)": au["excluded_tooth_a_pct"],
    })
seg_stage1_df = pd.DataFrame(seg_rows).set_index("Reply")
seg_stage1_df.to_csv(OUT_ROOT / "akudental_seg_stage1_frozen.csv")
display(seg_stage1_df.round(4))

## Step 5 · `akudental_seg` Stages 2 & 3 — Converged 4-Arm 82-Param Learning Curve (`K = 1..200`, 3 Seeds)

Uses mini-batching (`batch_size=32`), alive ReLU init (`feature_bias=+0.1`), `lr=3e-2` with cosine decay (`epochs=150`), and incremental Drive checkpointing after every `(seed, K)` (~3–4 min on T4 GPU).

In [ ]:
import matplotlib.pyplot as plt
from wsp_core.training.runner import plot_segmentation_learning_curve, run_segmentation_three_arm_validation

train_seg = load_akudental_seg_public(DATA_DIR, split="train")
SEG_OUT = OUT_ROOT / "akudental_seg_converged"
seg_runs, seg_summary = run_segmentation_three_arm_validation(
    task="akudental_seg",
    architecture="pano-wsp82-v1",
    train_samples=train_seg,
    val_samples=val_seg,
    design_lock_path=seg_design,
    responses_lock_path=seg_resp,
    budgets=[1, 2, 4, 8, 16, 32, 64, len(train_seg)],
    seeds=[0, 1, 2],
    epochs=150,
    lr=3e-2,
    batch_size=32,
    cosine_lr=True,
    device=DEVICE,
    out_dir=SEG_OUT,
    resume=True,
)
display(seg_summary.round(4))
fig = plot_segmentation_learning_curve(seg_summary, task="akudental_seg")
plt.show()
plt.close(fig)
print(f"Saved converged akudental_seg CSVs, representative payloads, and PNG to {SEG_OUT}")

## Step 6 · Visual X-Ray Comparison 1 — All 6 Frozen Replies (`A · Intact` vs. `B · Image-Only`)

In [ ]:
import matplotlib.pyplot as plt
from wsp_core.visualization import plot_frozen_replies_grid, select_deterministic_examples

ATTR = "Dataset: AKUDENTAL (Oz et al., BMC Oral Health 2025, CC BY-NC-SA 4.0) · Deterministic validation sample"
VIZ_DIR = OUT_ROOT / "akudental_seg_visuals"
examples = select_deterministic_examples(val_seg, n=2)

for idx, ex in enumerate(examples, start=1):
    out_p = VIZ_DIR / f"example_{idx}_frozen_6replies.png"
    plot_frozen_replies_grid(
        ex,
        frozen_seg,
        title=f"AKUDENTAL Validation Example {idx} · All 6 Frozen pano-wsp82-v1 Replies (A Intact vs. B Image-Only)",
        attribution=ATTR,
        out_path=out_p,
    )
    img = plt.imread(out_p)
    plt.figure(figsize=(11, 12)); plt.imshow(img); plt.axis("off"); plt.show()
print(f"Saved 6-reply X-ray overlays to {VIZ_DIR}")

## Step 7 · Visual X-Ray Comparison 2 — All 4 Arms Side-by-Side on the Same X-Rays (`K = 200`)

In [ ]:
import json
import matplotlib.pyplot as plt
from wsp_core.visualization import plot_four_arm_comparison

rep_path = OUT_ROOT / "akudental_seg_converged/akudental_seg_representative_payloads.json"
assert rep_path.is_file(), f"Run Step 5 first to generate {rep_path}"
rep_payloads = json.loads(rep_path.read_text(encoding="utf-8"))

for idx, ex in enumerate(examples, start=1):
    out_p = VIZ_DIR / f"example_{idx}_four_arms.png"
    plot_four_arm_comparison(
        ex,
        rep_payloads,
        title=f"AKUDENTAL Validation Example {idx} · 4-Arm Comparison (82-Parameter pano-wsp82-v1)",
        attribution=ATTR,
        out_path=out_p,
    )
    img = plt.imread(out_p)
    plt.figure(figsize=(12, 7)); plt.imshow(img); plt.axis("off"); plt.show()
print(f"Saved 4-arm X-ray comparison overlays to {VIZ_DIR}")